In [1]:
# Core frameworks
!pip install langgraph langchain langchain-groq

# Natural Language Processing (for any manual fallback)
!pip install spacy

# Download the English model for spaCy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
# --- 1. Core Logic & Typing ---
import os
import json
import re
from typing import TypedDict, List, Optional
from getpass import getpass

# --- 2. LangGraph & Orchestration ---
from langgraph.graph import StateGraph, END

# --- 3. Groq & LangChain (The "Thinking" Layer) ---
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# --- 4. NLP (The "Cleanup" Layer) ---
import spacy
nlp = spacy.load("en_core_web_sm")

In [2]:
# Securely set your Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

# Define the State for Project 2
class FinancialState(TypedDict):
    report_text: str
    extracted_old_val: float
    extracted_new_val: float
    ai_claimed_growth: float
    actual_growth: float
    is_math_correct: bool
    error_margin: float
    status_message: str
    summary: Optional[str] # To store the final corrected output

Enter your Groq API Key: ··········


In [3]:
import json
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

def groq_extractor_node(state):
    print("---LOG: Groq is Extracting Financial Data---")

    llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

    # We force the LLM to return a very specific format
    prompt = ChatPromptTemplate.from_template("""
    Extract the following financial figures from the text.
    Return ONLY a JSON object with these keys: 'old_val', 'new_val', 'claimed_growth'.
    If a value is missing, use 0.0.

    Text: {text}

    JSON Output:
    """)

    chain = prompt | llm
    response = chain.invoke({"text": state["report_text"]})

    # Parse the string output into a Python Dictionary
    try:
        # We strip any markdown formatting just in case
        clean_json = response.content.replace("```json", "").replace("```", "").strip()
        data = json.loads(clean_json)

        return {
            "extracted_old_val": float(data.get("old_val", 0)),
            "extracted_new_val": float(data.get("new_val", 0)),
            "ai_claimed_growth": float(data.get("claimed_growth", 0))
        }
    except Exception as e:
        print(f"Extraction Error: {e}")
        return {"status_message": "⚠️ Groq failed to parse the financial data."}

In [4]:
def financial_calculator_node(state):
    print("---LOG: Calculating Actual Growth---")

    old = state["extracted_old_val"]
    new = state["extracted_new_val"]

    # 1. Handle the Zero Math Exception
    if old == 0:
        return {
            "actual_growth": 0.0,
            "is_math_correct": False,
            "status_message": "⚠️ Calculation Error: The previous year's value was zero. Growth cannot be calculated."
        }

    # 2. Smooth Calculation
    # Formula: ((New - Old) / Old) * 100
    calc_growth = ((new - old) / old) * 100

    return {
        "actual_growth": round(calc_growth, 2), # Round to 2 decimal places
        "status_message": "Success"
    }

In [5]:
def math_auditor_node(state):
    print("---LOG: Auditing AI Claims vs Python Reality---")

    ai_claim = state["ai_claimed_growth"]
    python_calc = state["actual_growth"]

    # 1. Absolute Difference
    diff = abs(ai_claim - python_calc)

    # 2. Set the Threshold (Error Margin)
    # 0.05 is a standard financial tolerance for rounding
    threshold = 0.05

    is_correct = diff <= threshold

    # 3. Final State Update
    if is_correct:
        status = "✅ MATH VERIFIED: The AI claim matches the calculation."
    else:
        status = f"❌ MATH ERROR: AI claimed {ai_claim}%, but actual is {python_calc}%."

    return {
        "is_math_correct": is_correct,
        "error_margin": round(diff, 4),
        "status_message": status
    }

In [6]:
def financial_router(state):
    # 1. Check for the "Zero Division" or "Missing Data" flag
    if "⚠️" in state["status_message"]:
        return "data_alert_node"

    # 2. Check if the math failed the audit
    if not state["is_math_correct"]:
        return "correction_node"

    # 3. Everything is perfect
    return END

# --- THE CORRECTION NODE ---
def correction_node(state):
    print("---LOG: Injecting Corrected Math into Final Output---")
    correct_val = state["actual_growth"]

    # We explicitly rewrite the summary to be 100% accurate
    new_summary = f"Audit Correction: The actual growth is {correct_val}%. (Note: Original AI claim was inaccurate)."

    return {"summary": new_summary}

In [8]:
from langgraph.graph import StateGraph, END

# 1. Initialize the Graph with our State definition
workflow = StateGraph(FinancialState)

# 2. Add all the Nodes we've built
workflow.add_node("extractor", groq_extractor_node)
workflow.add_node("calculator", financial_calculator_node)
workflow.add_node("auditor", math_auditor_node)
workflow.add_node("correction", correction_node)

# 3. Define the Flow (The Edges)
workflow.set_entry_point("extractor")
workflow.add_edge("extractor", "calculator")
workflow.add_edge("calculator", "auditor")

# 4. The Logic Gate (Conditional Edge)
workflow.add_conditional_edges(
    "auditor",
    financial_router, # Our router function from earlier
    {
        "correction_node": "correction",
        "data_alert_node": END, # Stop if data is bad
        END: END                # Stop if math is already correct
    }
)

# 5. Connect the correction node to the end
workflow.add_edge("correction", END)

# Compile
app = workflow.compile()

In [9]:
# --- THE TEST RUN ---

# 1. Define the "Bad" Input
test_report = """
The company reported a strong fiscal year.
Revenue in 2024 was $100.0M, which grew to $150.0M in 2025.
The AI analysis suggests this represents a 60% growth in annual revenue.
"""

# 2. Initialize State
inputs = {
    "report_text": test_report,
    "extracted_old_val": 0.0,
    "extracted_new_val": 0.0,
    "ai_claimed_growth": 0.0,
    "actual_growth": 0.0,
    "is_math_correct": False,
    "error_margin": 0.0,
    "status_message": ""
}

# 3. Run the Graph
print("Starting Financial Audit...")
final_output = app.invoke(inputs)

print("\n--- FINAL AUDIT REPORT ---")
print(f"Status: {final_output['status_message']}")
print(f"Final Summary: {final_output['summary'] if 'summary' in final_output else 'No correction needed.'}")

Starting Financial Audit...
---LOG: Groq is Extracting Financial Data---
---LOG: Calculating Actual Growth---
---LOG: Auditing AI Claims vs Python Reality---
---LOG: Injecting Corrected Math into Final Output---

--- FINAL AUDIT REPORT ---
Status: ❌ MATH ERROR: AI claimed 60.0%, but actual is 50.0%.
Final Summary: Audit Correction: The actual growth is 50.0%. (Note: Original AI claim was inaccurate).
